In [ ]:
using JuMP
using Gurobi
using Random
using Dualization
using Plots
using DataFrames
using CSV
import XLSX
import JSON

current_directory = @__DIR__
functions_directory = joinpath(current_directory, "functions")
data_dir = joinpath(current_directory, "data")
results_dir = joinpath(current_directory, "results")

# Include all the function files
# include(joinpath(functions_directory, "create_check_params.jl"))
# include(joinpath(functions_directory, "deterministic_equivalent.jl"))
# include(joinpath(functions_directory, "generate_cuts_from_dual.jl"))
include(joinpath(functions_directory, "load_model_starting_points.jl"))
# include(joinpath(functions_directory, "load_model_starting_points_OLD.jl"))
include(joinpath(functions_directory, "initialize_parameters.jl"))
# include(joinpath(functions_directory, "process_scenario_data_n_selected_with_MVP2.jl"))
include(joinpath(functions_directory, "process_scenario_data_n_selected_with_MVP2_demand_scaling.jl"))
include(joinpath(functions_directory, "process_scenario_data_n_selected.jl"))
# include(joinpath(functions_directory, "save_L_shaped_results.jl"))
include(joinpath(functions_directory, "select_random_scenarios.jl"))
include(joinpath(functions_directory, "create_vaccine_data.jl"))
# include(joinpath(functions_directory, "sub_problem.jl"))
# include(joinpath(functions_directory, "master_problem.jl"))
# include(joinpath(functions_directory, "create_vaccine_data_MMR_only.jl"))

create_vaccine_data

In [ ]:
using Random

num_trials = 5



println("Seed Sequence: BASE_OFFSET ($BASE_OFFSET) + SUM of Increases (starting at 1, doubling)\n")

for trial in 1:num_trials
    
    # 1. Set the seed for the current trial
    Random.seed!(current_seed)

    println("--- Starting Trial $trial ---")
    println("Seed: $current_seed (Increase: $current_increase)")

    # 2. Generate random data
    random_sample = rand(1:100, 3) 
    println("Generated sample: $random_sample\n")


end

Seed Sequence: BASE_OFFSET (10) + SUM of Increases (starting at 1, doubling)

--- Starting Trial 1 ---
Seed: 20 (Increase: 10)
Generated sample: [29, 78, 72]

--- Starting Trial 2 ---
Seed: 40 (Increase: 20)
Generated sample: [51, 100, 68]

--- Starting Trial 3 ---
Seed: 80 (Increase: 40)
Generated sample: [87, 6, 52]

--- Starting Trial 4 ---
Seed: 160 (Increase: 80)
Generated sample: [73, 44, 6]

--- Starting Trial 5 ---
Seed: 320 (Increase: 160)
Generated sample: [46, 26, 61]



In [ ]:
A, V, A_v, P, P_v, V_a, V_p, P_a, A_p, capacity_category, vaccine_category, antigen_category = create_vaccine_data()

(["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"], ["TT", "HepB", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"], Dict("Td" => ["Diphtheria", "Tetanus"], "PCV" => ["PCV"], "Rotavirus" => ["Rotavirus"], "DTwP-Hib" => ["Diphtheria", "Tetanus", "Pertussis", "Hib"], "IPV" => ["Polio"], "Hexa" => ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"], "TT" => ["Tetanus"], "HPV" => ["HPV"], "DT" => ["Diphtheria", "Tetanus"], "HepB" => ["Hepatitis_B"]…), ["AJ_Vaccines", "BB_NCIPD", "China_National", "Bharat_Biotech", "Bilthoven", "Biological_E", "GSK", "Haffkine_Bio", "LG_Chem", "Merck_Sharp", "Panacea_Biotec", "PT_Bio", "Sanofi", "Serum_Institute", "Pfizer"], Dict("Td" => ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"], "PCV" => ["Serum_Institute", "GSK", "Pfizer"], "Rotavirus" => ["Serum_Institute", "GSK", "Bharat_Biotech"], "DTwP-Hib" => ["Serum_Institute"], "IPV" => ["LG_Che

In [ ]:
starting_points_vect_F, starting_points_vect_I, starting_points_vect_S = load_model_starting_points(data_dir, 1, 1, A, V)

(Any[("Diphtheria", 1, 1), ("Tetanus", 1, 1), ("Pertussis", 1, 1), ("Hib", 1, 1), ("Hepatitis_B", 1, 1), ("Polio", 1, 1), ("Rotavirus", 1, 1), ("PCV", 1, 3), ("HPV", 1, 5)], Any[("Penta", 3.980561078e8), ("OPV", 2.796866021e7), ("IPV", 4.1012800142e8), ("PCV", 9.87633795089999e15), ("TT", 7.6826e6), ("HepB", 3.2789883e6), ("DT", 1.1683437835e8), ("Td", 5.29899759e6), ("DTwP", 3.029039439e7), ("DTwP-Hib", 4.327199198e7), ("Hexa", 8.654398396e7), ("HPV", 4.42285505e8), ("Rotavirus", 1.567269e8)], Any[("Diphtheria", 0.0), ("Hib", 0.0), ("PCV", 0.0), ("Pertussis", 0.0), ("Polio", 0.0), ("Rotavirus", 0.0), ("Tetanus", 0.0), ("Hepatitis_B", 0.0), ("HPV", 0.0)])

In [ ]:
T, T_initial, Δ, s_real, r, r_avg, r_producer_avg, g, h, l, f_profit, Γ, F_time_set, κ, L_lower_number, L_upper_number, delta, beta, zeta_vm, phi_vm_lower, phi_vm_upper, m_segments = initialize_parameters(data_dir, 1, 1, 10, 5, P, V, P_v, V_p, 1)

([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 2, 3, 4, 5], Dict{Any, Any}("Sanofi" => 9.73e6, "Pfizer" => 5.551e7, "AJ_Vaccines" => 5.579e8, "Serum_Institute" => 5.201e6, "China_National" => 4.354e7, "BB_NCIPD" => 1.183e7, "Merck_Sharp" => 9.8e6, "Bilthoven" => 6.335e7, "Haffkine_Bio" => 8.12e6, "Biological_E" => 5.327e7…), Dict{Any, Any}(("PCV", "Serum_Institute", 6) => 2.29375, ("Rotavirus", "Bharat_Biotech", 2) => 0.9874999999999999, ("Rotavirus", "Serum_Institute", 2) => 0.9, ("Rotavirus", "Serum_Institute", 3) => 0.9, ("PCV", "GSK", 10) => 2.950390625, ("Rotavirus", "Bharat_Biotech", 10) => 0.9249999999999999, ("IPV", "Bilthoven", 5) => 1.9058333333333335, ("Rotavirus", "GSK", 10) => 2.1732915, ("DT", "BB_NCIPD", 3) => 0.175, ("OPV", "Serum_Institute", 9) => 0.13…), Dict{Any, Any}(("DT", 10) => 0.16999999999999998, ("IPV", 3) => 1.812625, ("DTwP", 2) => 0.177, ("HepB", 9) => 0.42874999999999996, ("Rotavirus", 7) => 1.3549595555555554, ("OPV", 3) => 0.13

In [ ]:
# Scenarios_used, p_ω_test, p_ω_test_partial_2, Ω_test_partial_1, Ω_test_partial_2, partial_scenario, s_real_tilde, d_real_tilde, random_scenarios = process_scenario_data_n_selected_with_MVP2(current_directory, data_dir, 3, 3, A, T, P, 1, 10, 5, 1, 1, 1, 1,  true, 1, 19)
random_scenarios, s_real_tilde, d_real_tilde = process_scenario_data_n_selected_with_MVP2_demand_scaling(current_directory, data_dir, 1, 1, A, T, P, 1, 10, 5, 1, 1, 1, 1, true, 1, 0.02, 9)

([1], Dict{Any, Any}(("GSK", 6, 1) => 2.815e7, ("Panacea_Biotec", 10, 1) => 4.539e8, ("Serum_Institute", 7, 1) => 8.33e6, ("China_National", 10, 1) => 2.996e7, ("Biological_E", 6, 1) => 5.549e7, ("Bharat_Biotech", 5, 1) => 7.7e7, ("BB_NCIPD", 8, 1) => 2.702e7, ("LG_Chem", 5, 1) => 2.779e8, ("PT_Bio", 2, 1) => 6.56e7, ("China_National", 1, 1) => 4.354e7…), Dict{Any, Any}(("Pertussis", 5, 1) => 3.8516391342e8, ("Hepatitis_B", 5, 1) => 7.629096566e7, ("Pertussis", 2, 1) => 4.2101715871e8, ("HPV", 7, 1) => 6.506605949e7, ("PCV", 5, 1) => 3.2002154181e8, ("Pertussis", 8, 1) => 4.7583252275e8, ("Hepatitis_B", 1, 1) => 5.950255214e7, ("Diphtheria", 4, 1) => 6.8381413097e8, ("Hib", 3, 1) => 9.5634917861e8, ("Diphtheria", 2, 1) => 7.0625691759e8…))

In [ ]:
for trial in 1:5
    seed = (rand(1:9999))
    random_scenarios, s_real_tilde, d_real_tilde = process_scenario_data_n_selected_with_MVP2_demand_scaling(current_directory, data_dir, 1, 1, A, T, P, 1, 10, 5, 1, 1, 1, 1, true, 1, 0.02, seed)
    println(d_real_tilde)
end

Dict{Any, Any}(("Pertussis", 5, 1) => 5.2960038095e8, ("Hepatitis_B", 5, 1) => 8.923318305e7, ("Pertussis", 2, 1) => 4.4956069489e8, ("HPV", 7, 1) => 6.81024756e7, ("PCV", 5, 1) => 3.0186429121e8, ("Pertussis", 8, 1) => 3.8134095795e8, ("Hepatitis_B", 1, 1) => 5.950255214e7, ("Diphtheria", 4, 1) => 7.021305809e8, ("Hib", 3, 1) => 7.9383232473e8, ("Diphtheria", 2, 1) => 9.8621461465e8, ("Tetanus", 4, 1) => 7.0402873905e8, ("Rotavirus", 1, 1) => 1.4247900337e8, ("PCV", 8, 1) => 3.571258111e8, ("Hepatitis_B", 6, 1) => 7.63312537e7, ("Hepatitis_B", 9, 1) => 1.0194697924e8, ("PCV", 6, 1) => 2.9477942184e8, ("HPV", 5, 1) => 1.2467625212e8, ("Tetanus", 7, 1) => 7.1010331216e8, ("Rotavirus", 6, 1) => 3.1258331039e8, ("Tetanus", 5, 1) => 7.6880305936e8, ("Polio", 2, 1) => 7.7727156505e8, ("HPV", 9, 1) => 4.399431063e7, ("Polio", 7, 1) => 1.7466329128e8, ("Hib", 9, 1) => 1.5214205103e8, ("Hib", 1, 1) => 5.2218200889e8, ("Polio", 3, 1) => 7.4382713892e8, ("Hib", 2, 1) => 7.234604567e8, ("Diphther

In [ ]:
    scenario_pairs_path = joinpath(data_dir, "scenario_pairs_new_1_scenario_6OCT_NO_MMR.json")
    scenario_pairs = JSON.parsefile(scenario_pairs_path)

    demand_dict = Dict()
    d_real_tilde = Dict()

        total_demand = 0.0
        for a in A
            # new_rate = round((0.6 + 0.9 * rand()), digits=2) # demand_growth_rate)
            for t in T
                growth_multiplier = t == 1 ? 1.0 : round((1.10 + 0.5 * rand()), digits=2) # demand_growth_rate)
                d_real_tilde[a, t, 1] = round(
                    (scenario_pairs["1"]["demand"]["$t"]["$a"] * growth_multiplier) / 1,
                    digits=2,
                )
                total_demand += d_real_tilde[a, t, 1]
            end
        end
        demand_dict[1] = total_demand


    print(d_real_tilde)

Dict{Any, Any}(("Pertussis", 5, 1) => 3.9891976747e8, ("Hepatitis_B", 5, 1) => 9.059552172e7, ("Pertussis", 2, 1) => 3.9604156455e8, ("HPV", 7, 1) => 6.333096457e7, ("PCV", 5, 1) => 3.2683051078e8, ("Pertussis", 8, 1) => 4.8258192024e8, ("Hepatitis_B", 1, 1) => 5.950255214e7, ("Diphtheria", 4, 1) => 8.1202928052e8, ("Hib", 3, 1) => 7.7508038005e8, ("Diphtheria", 2, 1) => 8.6532379092e8, ("Tetanus", 4, 1) => 7.4108288321e8, ("Rotavirus", 1, 1) => 1.4247900337e8, ("PCV", 8, 1) => 2.7454046728e8, ("Hepatitis_B", 6, 1) => 1.0487250508e8, ("Hepatitis_B", 9, 1) => 9.661641823e7, ("PCV", 6, 1) => 3.3978391373e8, ("HPV", 5, 1) => 1.2277279789e8, ("Tetanus", 7, 1) => 7.9580543605e8, ("Rotavirus", 6, 1) => 3.3581585373e8, ("Tetanus", 5, 1) => 7.5650221041e8, ("Polio", 2, 1) => 9.088098299e8, ("HPV", 9, 1) => 5.23081961e7, ("Polio", 7, 1) => 1.6257121727e8, ("Hib", 9, 1) => 2.0061208499e8, ("Hib", 1, 1) => 5.2218200889e8, ("Polio", 3, 1) => 9.1884528925e8, ("Hib", 2, 1) => 8.011876132e8, ("Diphth

In [ ]:
f_profit
# 1. Initialize empty vectors to hold the data for each column
vaccine = []
manufacturer = []
val1 = []
val2 = []
price = [] # Or 'value', 'quantity', etc., based on what the number represents

# 2. Iterate over the key-value pairs in the dictionary and populate the vectors
for (key_tuple, value) in f_profit
    # The key_tuple is structured as: ("VaccineType", "Manufacturer", (Int1, Int2))
    push!(vaccine, key_tuple[1])         # "PCV"
    push!(manufacturer, key_tuple[2])    # "Pfizer"
    
    # The third element is a nested tuple, (Int1, Int2)
    nested_tuple = key_tuple[3]
    push!(val1, nested_tuple[1])         # 1
    push!(val2, nested_tuple[2])         # 5
    
    push!(price, value)                  # 956.547
end

# 3. Create the DataFrame from the vectors
df = DataFrame(
    Vaccine = vaccine,
    Manufacturer = manufacturer,
    Val_1 = val1,
    Val_2 = val2,
    Price = price
)

# Display the resulting DataFrame
println(df)

file_path_simple = "F_output.xlsx"
XLSX.writetable(file_path_simple, df)

1680×5 DataFrame
  Row │ Vaccine    Manufacturer     Val_1  Val_2  Price    
      │ Any        Any              Any    Any    Any      
──────┼────────────────────────────────────────────────────
    1 │ PCV        Pfizer           1      5      956.547
    2 │ OPV        Panacea_Biotec   3      7      0.941018
    3 │ Penta      Serum_Institute  4      4      1.77333
    4 │ TT         PT_Bio           2      3      2.06088
    5 │ TT         BB_NCIPD         6      7      0.877644
    6 │ Penta      Biological_E     2      4      11.2447
    7 │ HPV        GSK              8      9      17.9414
    8 │ Penta      Panacea_Biotec   2      2      17.2837
    9 │ PCV        Serum_Institute  1      1      0.102041
   10 │ DTwP       Biological_E     4      8      9.66587
   11 │ Penta      Serum_Institute  4      7      1.77333
   12 │ IPV        Bilthoven        5      8      185.907
   13 │ HPV        China_National   6      6      12.0219
   14 │ Td         Biological_E     10     10 

In [ ]:
using JSON

file_path = normpath(joinpath(pwd(), "results/2 segments/SB", 
    "MVP_DE_results_T_10_delta_5_scen_1_trial_1_inv_1_cap._1_cap.inc._1.json"))

println(file_path)

data = JSON.parsefile(file_path)

c:\Users\Nicholas Uhorchak\OneDrive - University of Arkansas\Desktop\Vaccine_Tender\Deterministic\Model Files\Segment Test\results\2 segments\SB\MVP_DE_results_T_10_delta_5_scen_1_trial_1_inv_1_cap._1_cap.inc._1.json


Dict{String, Any} with 10 entries:
  "Y"  => Dict{String, Any}("Sanofi"=>Dict{String, Any}("3"=>1.0, "4"=>1.0, "1"…
  "Q"  => Dict{String, Any}("PCV"=>Dict{String, Any}("Serum_Institute"=>Dict{St…
  "I"  => Dict{String, Any}("PCV"=>Dict{String, Any}("4"=>Dict{String, Any}("1"…
  "W"  => Dict{String, Any}("Sanofi"=>Dict{String, Any}("3"=>Dict{String, Any}(…
  "X"  => Dict{String, Any}("PCV"=>Dict{String, Any}("Serum_Institute"=>Dict{St…
  "S"  => Dict{String, Any}("Hib"=>Dict{String, Any}("4"=>Dict{String, Any}("1"…
  "Z"  => Dict{String, Any}("PCV"=>Dict{String, Any}("Serum_Institute"=>Dict{St…
  "L"  => Dict{String, Any}("Sanofi"=>Dict{String, Any}("3"=>1.0, "4"=>1.0, "1"…
  "Vc" => Dict{String, Any}("PCV"=>Dict{String, Any}("3"=>Dict{String, Any}("1"…
  "F"  => Dict{String, Any}("Hib"=>Dict{String, Any}("3"=>Dict{String, Any}("4"…

In [ ]:
#python
import json, pandas as pd

with open("results/2 segments/SB/MVP_DE_results_T_10_delta_5_scen_1_trial_5_inv_1_cap._1_cap.inc._1.json") as f:
    data = json.load(f)

rows = []
for vaccine, vdict in data["Q"].items():
    for producer, pdict in vdict.items():
        for t, tdict in pdict.items():
            for tau, taudict in tdict.items():
                for segment, val in taudict.items():
                    rows.append({
                        "vaccine": vaccine,
                        "producer": producer,
                        "t": int(t),
                        "tau": int(tau),
                        "segment": int(segment),
                        "value": val
                    })

Q_df = pd.DataFrame(rows)


In [ ]:
Q_df.to_excel("Q_data_flat.xlsx", index=False)

### inter-set comparisons

In [ ]:
using JSON, Statistics, Printf, DataFrames, LinearAlgebra

#compare within sets

# === Load and flatten schedules ===
function flatten_F(F)
    Dict((ant, s, e) => v for (ant, sub) in F for (s, ends) in sub for (e, v) in ends)
end

# === Extract the set of active intervals (for Jaccard) ===
function active_set(F)
    Set((ant, s, e) for (ant, sub) in F for (s, ends) in sub for (e, v) in ends if v == 1.0)
end

function compare(F1, F2)
    f1, f2 = flatten_F(F1), flatten_F(F2)
    keys_union = union(keys(f1), keys(f2))
    vec1 = [get(f1, k, 0.0) for k in keys_union]
    vec2 = [get(f2, k, 0.0) for k in keys_union]

    hamming = mean(vec1 .!= vec2)  #sum(vec1 .!= vec2)
    set1, set2 = active_set(F1), active_set(F2)
    jaccard = length(intersect(set1, set2)) / max(length(union(set1, set2)), 1)
    cosine = 1 - dot(vec1, vec2) / (norm(vec1) * norm(vec2) + eps())

    return hamming, jaccard, cosine
end

# === Load all trials ===
files = [joinpath("results/3 segments/MP",
    "MVP_DE_results_T_10_delta_5_scen_1_trial_$(i)_inv_1_cap._1_cap.inc._1.json") for i in 1:5]

trials = [JSON.parsefile(f)["F"] for f in files]

# === Build distance/similarity matrices ===
n = length(trials)
hamming_matrix = zeros(Float64, n, n)
jaccard_matrix = zeros(Float64, n, n)
cosine_matrix = zeros(Float64, n, n)

for i in 1:n, j in i+1:n
    h, jacc, cosd = compare(trials[i], trials[j])
    hamming_matrix[i,j] = hamming_matrix[j,i] = h
    jaccard_matrix[i,j] = jaccard_matrix[j,i] = jacc
    cosine_matrix[i,j] = cosine_matrix[j,i] = cosd
end

for i in 1:n
    jaccard_matrix[i,i] = 1.0
    # cosine_matrix[i,i] = 0.0  # distance between identical vectors
end

println("Pairwise Hamming distances:")
for row in eachrow(hamming_matrix)
    @printf("%s\n", join(row, "\t"))
end

println("Pairwise Jaccard distances:")
for row in eachrow(jaccard_matrix)
    for val in row
        @printf("%.3f\t", val)
    end
    println()
end

println("\nPairwise Cosine distances:")
for row in eachrow(cosine_matrix)
    @printf("%s\n", join(round.(row, digits=3), "\t"))
end

# === Summary statistics ===
function summarize_matrix(M; name="")
    vals = Float64[]
    for i in 1:size(M,1)
        for j in i+1:size(M,2)
            push!(vals, M[i,j])
        end
    end

    if isempty(vals)
        println("\n$name summary: only one element, skipping.")
        return
    end

    println("\n$name summary:")
    println("  min:  ", round(minimum(vals), digits=4))
    println("  max:  ", round(maximum(vals), digits=4))
    println("  mean: ", round(mean(vals), digits=4))
end

summarize_matrix(hamming_matrix, name="Hamming")
summarize_matrix(jaccard_matrix, name="Jaccard")
summarize_matrix(cosine_matrix, name="Cosine")



Pairwise Hamming distances:
0.0	0.17777777777777778	0.2111111111111111	0.24722222222222223	0.23055555555555557
0.17777777777777778	0.0	0.20555555555555555	0.16944444444444445	0.18055555555555555
0.2111111111111111	0.20555555555555555	0.0	0.18611111111111112	0.15833333333333333
0.24722222222222223	0.16944444444444445	0.18611111111111112	0.0	0.16666666666666666
0.23055555555555557	0.18055555555555555	0.15833333333333333	0.16666666666666666	0.0
Pairwise Jaccard distances:
1.000	0.439	0.367	0.299	0.331	
0.439	1.000	0.383	0.465	0.440	
0.367	0.383	1.000	0.427	0.491	
0.299	0.465	0.427	1.000	0.474	
0.331	0.440	0.491	0.474	1.000	

Pairwise Cosine distances:
0.0	0.39	0.463	0.539	0.503
0.39	0.0	0.446	0.365	0.389
0.463	0.446	0.0	0.401	0.341
0.539	0.365	0.401	0.0	0.357
0.503	0.389	0.341	0.357	0.0

Hamming summary:
  min:  0.1583
  max:  0.2472
  mean: 0.1933

Jaccard summary:
  min:  0.2992
  max:  0.4911
  mean: 0.4115

Cosine summary:
  min:  0.3413
  max:  0.5393
  mean: 0.4196


### set comparisons

In [ ]:
#compare UG/SB/MP sets
using JSON, Statistics, Printf, DataFrames, LinearAlgebra

function compare_sets(setA, setB)
    nA, nB = length(setA), length(setB)
    hamming_vals, jaccard_vals, cosine_vals = Float64[], Float64[], Float64[]

    for i in 1:nA, j in 1:nB
        h, jacc, cosd = compare(setA[i], setB[j])
        push!(hamming_vals, h)
        push!(jaccard_vals, jacc)
        push!(cosine_vals, cosd)
    end

    return mean(hamming_vals), mean(jaccard_vals), mean(cosine_vals)
end

# segment_paths = [
#     "results/3 segments/UG",
#     "results/3 segments/SB",
#     "results/3 segments/MP"
# ]

segment_paths = [
    "results/1 segment/UG",
    "results/3 segments/UG"
]

sets = [
    [JSON.parsefile(joinpath(path,
        "MVP_DE_results_T_10_delta_5_scen_1_trial_$(i)_inv_1_cap._1_cap.inc._1.json"))["F"]
        for i in 1:5]
    for path in segment_paths
]

# names = ["UG", "SB", "MP"]
names = ["UG - No segments", "UG - 3 Segments"]
for (i, A) in enumerate(sets), (j, B) in enumerate(sets)
    i < j || continue
    h, jacc, cosd = compare_sets(A, B)
    @printf("Avg distances %s ↔ %s → Hamming: %.3f, Jaccard: %.3f, Cosine: %.3f\n",
        names[i], names[j], h, jacc, cosd)
end



Avg distances UG - No segments ↔ UG - 3 Segments → Hamming: 0.172, Jaccard: 0.151, Cosine: 0.727


### Z segment analysis

In [ ]:
using JSON, DataFrames, Statistics, Printf

segment_paths = [
    # "results/1 segment/UG",
    "results/1 segment/UG"
]

sets = [
    [JSON.parsefile(joinpath(path,
        "MVP_DE_results_T_10_delta_5_scen_1_trial_$(i)_inv_1_cap._1_cap.inc._1.json"))
        for i in 1:5]
    for path in segment_paths
]

1-element Vector{Vector{Dict{String, Any}}}:
 [Dict("Y" => Dict{String, Any}("Sanofi" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "Pfizer" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => -0.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "AJ_Vaccines" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => -0.0…), "Serum_Institute" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "China_National" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "BB_NCIPD" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" =

In [ ]:
function flatten_Z(Z; model_group::String="", trial::Int=0)
    rows = Dict{Symbol,Any}[]
    for (vaccine, producers) in Z
        for (producer, times) in producers
            for (t, segments) in times
                for (seg, val) in segments
                    push!(rows, Dict(
                        :vaccine => vaccine,
                        :producer => producer,
                        :time => parse(Int, t),
                        :segment => parse(Int, seg),
                        :Z => val,
                        :model_group => model_group,
                        :trial => trial
                    ))
                end
            end
        end
    end
    DataFrame(rows)
end

function flatten_Q(Q; model_group::String="", trial::Int=0)
    rows = Dict{Symbol,Any}[]
    for (vaccine, producers) in Q
        for (producer, starts) in producers
            for (t, ends) in starts
                for (τ, segments) in ends
                    for (seg, val) in segments
                        push!(rows, Dict(
                            :vaccine => vaccine,
                            :producer => producer,
                            :start_time => parse(Int, t),
                            :end_time => parse(Int, τ),
                            :segment => parse(Int, seg),
                            :Q => val,
                            :model_group => model_group,
                            :trial => trial
                        ))
                    end
                end
            end
        end
    end
    DataFrame(rows)
end


flatten_Q (generic function with 1 method)

In [ ]:
# --- Build Z and Q DataFrames safely ---
Z_df_list = []
Q_df_list = []

for g in eachindex(sets)
    for i in eachindex(sets[g])
        push!(Z_df_list, flatten_Z(sets[g][i]["Z"];
            model_group = basename(segment_paths[g]),
            trial = i))
        push!(Q_df_list, flatten_Q(sets[g][i]["Q"];
            model_group = basename(segment_paths[g]),
            trial = i))
    end
end

Z_df = vcat(Z_df_list...)
Q_df = vcat(Q_df_list...)

# --- Join and compute realized quantities ---
combined_df = leftjoin(Q_df, Z_df,
    on = [:vaccine, :producer, :start_time => :time, :segment, :model_group, :trial])

combined_df.realized_Q = combined_df.Q .* combined_df.Z

8400-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [ ]:
using XLSX

XLSX.writetable("Z_and_Q_UG_1_segment.xlsx", Tables.columntable(combined_df); sheetname="Sheet1", overwrite=true)

### find summary stats and help fund Z bins

In [ ]:
import json, pandas as pd, numpy as np

# Load file
with open("results/2 segments/UG/MVP_DE_results_T_10_delta_5_scen_1_trial_3_inv_1_cap._1_cap.inc._1.json") as f:
    data = json.load(f)

# Flatten deeply nested structure under 'Q' and collect all numeric values
def extract_numbers(d):
    vals = []
    if isinstance(d, dict):
        for v in d.values():
            vals += extract_numbers(v)
    elif isinstance(d, (int, float)):
        vals.append(d)
    return vals

q_vals = extract_numbers(data["Q"])
series = pd.Series(q_vals)

# Summary statistics
print(series.describe())

count    1.680000e+03
mean     7.381764e+06
std      6.524467e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.666683e+09
dtype: float64


In [ ]:
import json, pandas as pd, numpy as np

# Load data
with open("results/2 segments/UG/MVP_DE_results_T_10_delta_5_scen_1_trial_3_inv_1_cap._1_cap.inc._1.json") as f:
    data = json.load(f)

# Recursively extract all numbers under "Q"
def extract_numbers(d):
    if isinstance(d, dict): return sum((extract_numbers(v) for v in d.values()), [])
    return [d] if isinstance(d, (int, float)) else []

q_vals = pd.Series(extract_numbers(data["Q"]))

# Function to create n breakpoints
def create_breakpoints(values, n):
    vals = values[values > 0]
    if len(vals) == 0: return None, None

    # log-space edges from smallest positive value to max, plus 0 and ∞
    edges = np.logspace(np.log10(vals.min()), np.log10(vals.max()), n - 1)
    edges = np.unique(np.round(np.concatenate(([0], edges, [1e19])), 0))
    bins = pd.cut(values, bins=edges, include_lowest=True)
    labels = [f"{e:.0e}" for e in edges]
    return bins, labels


# Example: split into 5 bins
bins, labels = create_breakpoints(q_vals, 6)
print("Bin edges:", labels)
print(bins.value_counts().sort_index())

Bin edges: ['0e+00', '1e+02', '6e+03', '4e+05', '3e+07', '2e+09', '1e+19']
(-0.001, 99.0]                1578
(99.0, 6328.0]                   6
(6328.0, 405630.0]               1
(405630.0, 26001082.0]          38
(26001082.0, 1666683369.0]      56
(1666683369.0, 1e+19]            1
Name: count, dtype: int64


### end of period missed doses

In [ ]:
using JSON, DataFrames

segment_paths = [
    # "results/1 segment/SB"
    "results/3 segments controlled random seed/SB"
]

sets = [
    [JSON.parsefile(joinpath(path,
        "MVP_DE_results_T_10_delta_5_scen_1_trial_$(i)_inv_1_cap._1_cap.inc._1.json"))
        for i in 1:5]
    for path in segment_paths
]

1-element Vector{Vector{Dict{String, Any}}}:
 [Dict("Y" => Dict{String, Any}("Sanofi" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "Pfizer" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => -0.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => -0.0, "9" => -0.0, "8" => 1.0…), "AJ_Vaccines" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "Serum_Institute" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "China_National" => Dict{String, Any}("3" => -0.0, "4" => -0.0, "1" => 1.0, "5" => -0.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "9" => 1.0, "8" => 1.0…), "BB_NCIPD" => Dict{String, Any}("3" => 1.0, "4" => 1.0, "1" => 1.0, "5" => 1.0, "2" => 1.0, "6" => 1.0, "7" => 1.0, "10" => 1.0, "

In [ ]:
sets[1][1]["S"]["Hib"]["10"]["1"]

300078.96625999815

In [ ]:
using Statistics

function sum_time10_per_set(sets)
    set_sums = Float64[]
    for s in sets
        total = 0.0
        for subset in s
            if haskey(subset, "S")
                S = subset["S"]
                for (antigen, tdict) in S
                    if isa(tdict, Dict) && haskey(tdict, "10")
                        v10 = tdict["10"]
                        if isa(v10, Dict)
                            total += sum(values(v10))
                        elseif isnumeric(v10)
                            total += v10
                        end
                    end
                end
            end
        end
        push!(set_sums, total)
    end
    return mean(set_sums)
end

avg_sum = sum_time10_per_set(sets)
println("Average of sums at t=10: ", avg_sum)



Average of sums at t=10: 2.4513281131979987e7


In [3]:
import re
from collections import defaultdict

# File path
file_path = "results/3 segments controlled random seed/UG/UG_output_3_segments.txt"

# Regex to match lines like "F Objective value: 1.5078124368585988e6"
pattern = re.compile(r"(\w+)\s+Objective value:\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")

# Store values by metric
values = defaultdict(list)

# Read and extract values
with open(file_path, "r") as f:
    for line in f:
        match = pattern.search(line)
        if match:
            
            metric, val = match.groups()
            values[metric].append(float(val))

# Compute and display averages
for metric, nums in values.items():
    avg = sum(nums) / len(nums)
    print(f"{metric} Objective value: {avg:.15g}")
